**7. Qualidade de Dados:**

A avaliação de qualidade deve ser realizada nas tabelas Bronze e Prata. Não se deve afirmar que existem erros sem executar as consultas.

**7.2. Consulta de completude:**

In [0]:
%sql
SELECT
    COUNT(*) AS total_itens,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS pedidos_nulos,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS produtos_nulos,
    SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS quantidades_nulas,
    SUM(CASE WHEN net_sales IS NULL THEN 1 ELSE 0 END) AS receitas_nulas,
    SUM(CASE WHEN profit IS NULL THEN 1 ELSE 0 END) AS lucros_nulos
FROM workspace.ecommerce_silver.itens_pedido;


total_itens,pedidos_nulos,produtos_nulos,quantidades_nulas,receitas_nulas,lucros_nulos
397569,0,0,0,0,0


**7.3. Consulta de unicidade dos pedidos:**

In [0]:
%sql
SELECT
    order_id,
    COUNT(*) AS total_registros
FROM workspace.ecommerce_silver.pedidos
GROUP BY order_id
HAVING COUNT(*) > 1;


order_id,total_registros


**7.4. Consulta de duplicidade de itens:**

In [0]:
%sql
SELECT
    order_id,
    product_id,
    COUNT(*) AS total_registros
FROM workspace.ecommerce_silver.itens_pedido
GROUP BY order_id, product_id
HAVING COUNT(*) > 1;


order_id,product_id,total_registros
ORD-949264,PROD-000248,2
ORD-380303,PROD-000899,2
ORD-394235,PROD-000568,2
ORD-442450,PROD-000656,2
ORD-726178,PROD-000467,2
ORD-172413,PROD-000648,2
ORD-142265,PROD-000252,2
ORD-777786,PROD-000719,2
ORD-527658,PROD-000856,2
ORD-396841,PROD-001098,2


**7.5. Consulta de integridade referencial:**

In [0]:
%sql
SELECT
    COUNT(*) AS itens_sem_produto_cadastrado
FROM workspace.ecommerce_silver.itens_pedido i
LEFT JOIN workspace.ecommerce_silver.produtos p
    ON i.product_id = p.product_id
WHERE p.product_id IS NULL;


itens_sem_produto_cadastrado
0


In [0]:
%sql
SELECT
    COUNT(*) AS pedidos_sem_cliente_cadastrado
FROM workspace.ecommerce_silver.pedidos p
LEFT JOIN workspace.ecommerce_silver.clientes c
    ON p.customer_id = c.customer_id
WHERE c.customer_id IS NULL;


pedidos_sem_cliente_cadastrado
0


**7.6. Consulta de consistência financeira:**

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.itens_pedido
WHERE quantity <= 0
   OR unit_price < 0
   OR gross_sales < 0
   OR discount_amount < 0
   OR net_sales < 0
   OR product_cost < 0
   OR discount_percentage < 0
   OR discount_percentage > 1;


order_id,product_id,quantity,unit_price,discount_percentage,discount_amount,gross_sales,tax_amount,shipping_cost,net_sales,product_cost,profit,_source_file,_ingestion_timestamp


**7.7. Consulta de avaliações inválidas:**

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_silver.pedidos
WHERE customer_rating < 0
   OR customer_rating > 5;


order_id,order_date,order_timestamp,order_status,sales_channel,customer_id,customer_type,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,customer_lifetime_value,is_repeat_customer,customer_order_count,_source_file,_ingestion_timestamp
